# Urban Similarity Model Analysis — São Paulo and Chicago (`sp_chicago_model_v1`)

This notebook builds the cross-city urban similarity model over **173 units** (96 São Paulo districts and 77 Chicago Community Areas) with **Brás (SP:10)** as reference, following `analysis/model_analysis.ipynb`. It loads the 12 accepted feature families for both cities, puts every feature on a common scale (convention **C6**), computes the dissimilarity between every pair of units and reports:

1. **R1 — equal weights**, **R2 — PCA weights** (components explaining 90% of the variance) and **R3 — without correlated features** (|ρ| ≥ 0.70 pruned), under the primary **hybrid** scaling;
2. two extra steps for the scale-comparison question: the same three runs under **all-absolute** and **all-relative** scaling.

The main question is *which Chicago Community Areas most resemble Brás*, so every ranking is shown for the 77 Chicago areas as well as for all units. All rules were fixed before any cross-city value was computed ([MODEL_PLAN](../docs/chicago/MODEL_PLAN.md), cross-city section). The calculations live in `analysis/scripts/sp_chicago_model.py` and `analysis/scripts/chicago_model.py`.

## Math and techniques

**Transforms.** Natural log for strictly positive ratio-scale amounts (job, resident, street and parcel densities, transit access, building height, block elongation); identity for shares, indices and log-ratios.

**Comparing levels across cities (C6).** Each transformed column is standardized, $z = (t-\bar t)/s$ (population SD), and capped at ±3. The mean and SD come either from all 173 units (**absolute**: the numbers are compared as measured) or from the unit's own city (**relative**: a unit is compared by its position in its city).
* **Hybrid (primary)**: absolute where both cities use the same instrument validated in both — M1, M3, M4, M6 (Overture streets), B1 (Overture buildings), U3 (census residents), M7 (ground parcels); relative where the instruments differ — U2 (RAIS vs LODES), U4 (PTAL rises with route count), BV (GHSL reads higher in São Paulo), U6 (place coverage differs), U1 (declared vs observed use).
* **All-absolute** and **all-relative** apply one rule to every column.

Relative standardization removes a city-wide shift and stretch caused by an instrument; it does not fix an instrument that mis-ranks units *inside* a city.

**Composition (U6).** Aitchison distance on centred log-ratios of the five activity classes, unscaled; city-centred (relative) except in the all-absolute run.

**Blocks, calibration and distance.** Each family is a block (U6 is two: intensity and composition, half its budget each). For block $b$, $m_b(d,e) = \frac{1}{c_b}\sum_j (x_{dj}-x_{ej})^2$, divided by its median positive value over all 14,878 pairs. With equal family budgets $w_f = 1/12$:
$$D(d,e)=\sqrt{\frac{\sum_b w_{f(b)}\,s_b\,m_b(d,e)/\beta_b}{\sum_{b\ \text{available}} w_{f(b)}\,s_b}}$$
The denominator is 1 except for pairs involving the three units without U1 (O'Hare, Marsilac, Parelheiros), which use the families both units have.

**PCA (R2).** Principal components of the R1 geometry, computed from $D$ (equal to PCA of the weighted embedding when every unit has every family); the fewest components reaching 90% of the variance; Euclidean distance on unwhitened scores.

**Correlation pruning (R3).** Spearman among the scaled scalar columns over 173 units; while any pair has |ρ| ≥ 0.70, drop the column with the most such pairs (ties: higher mean |ρ|, then later position).

## Attribute data dictionary

**20 columns in 12 families**, the same definitions in both cities. Chicago and São Paulo sources differ where noted.

| Family | Column | Chicago source | São Paulo source | C6 (hybrid) |
|---|---|---|---|---|
| M1 street density | `m1_km_per_km2` | Overture streets 2026 | Overture streets 2026 | absolute |
| M3 block size | `ov_m3_wmedian_ln_m2` | Overture street faces | Overture street faces | absolute |
| M4 block shape | `ov_m4_wmedian_compactness`, `ov_m4_wmedian_elongation` | Overture street faces | Overture street faces | absolute |
| M6 street hierarchy | `m6_major_share` | Overture streets | Overture streets | absolute |
| M7 ground-parcel density | `m7_parcels_per_km2` | Cook 2024 ground parcels | GeoSampa fiscal lots | absolute |
| B1 footprint coverage | `B1_coverage_land` | Overture buildings | Overture buildings | absolute |
| U3 resident density | `u3_acs_land_km2` | ACS 2020–2024 | Census 2022 | absolute |
| BV vertical form | `bv_height_built_m` | GHSL (2018) | GHSL (2018) | relative |
| U2 job density | `u2_jobs_land_km2` | LODES 2023 | RAIS 2022 | relative |
| U4 transit access | `u4_ptai_avg_resident` | PTAL: CTA bus, 'L', Metra | PTAL: SPTrans bus, Metrô, CPTM | relative |
| U6 activity composition | `u6_log_intensity` + five `u6_clr_city_*` | Overture places + ACS dwellings | Overture places + CNEFE dwellings | relative |
| U1 land-use shares | `p_residential`, `p_industrial`, `p_institutional` | CMAP 2023 (observed) | IPTU 2026 (declared) | relative |

In [ ]:
import sys, json
from pathlib import Path
from math import pi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

base_dir = Path.cwd() if (Path.cwd() / 'config/sp_chicago_model_v1.json').exists() else Path.cwd() / 'analysis'
sys.path.insert(0, str(base_dir / 'scripts'))
import chicago_model as cm
import sp_chicago_model as xm

TARGET = xm.REFERENCE                 # Brás
OUT = xm.OUT; FIG = OUT / 'figures'; TAB = OUT / 'tables'
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(FIG / f'{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
# Frozen inputs: every table is verified against the SHA-256 recorded at acceptance.
contract, df, subs, names, city, families = xm.load_inputs()
ids = df.index.tolist(); n = len(ids)
chi = [u for u in ids if u.startswith('CHI')]; spd = [u for u in ids if u.startswith('SP')]
label = lambda u: f"{names[u]} ({city[u]})"
print(f"{contract['contract_version']}: {contract['status']}")
print(f"{n} units: {len(spd)} São Paulo, {len(chi)} Chicago; reference {label(TARGET)}.")
print("Missing (U1, available-case):", [label(u) for u in df.index[df.p_residential.isna()]])
display(df.assign(name=names, city=city).set_index(['name', 'city'], append=True).loc[[TARGET, 'CHI:32']])

## 1. Comparing levels across cities (C6)

The boxplots show each transformed column by city. Where the boxes sit at different heights, absolute scaling keeps that gap and relative scaling removes it. The table lists the city medians and the level each scaling uses.

In [ ]:
scalar_cols = [c for cs in families.values() for c in cs if c not in cm.CLR]
lv = {s: xm.levels(s) for s in ('hybrid', 'all_absolute', 'all_relative')}
med = pd.DataFrame({'São Paulo median': df.loc[spd, scalar_cols].median(), 'Chicago median': df.loc[chi, scalar_cols].median(),
                    'Brás': df.loc[TARGET, scalar_cols], 'hybrid': pd.Series(lv['hybrid'])[scalar_cols]})
med.round(3).to_csv(TAB / 'city_levels.csv'); display(med.round(3))

fig, axes = plt.subplots(3, 5, figsize=(18, 10)); axes = axes.ravel()
for ax, c in zip(axes, scalar_cols):
    t = cm.transform(df[c], cm.TRANSFORM[c])
    sns.boxplot(x=city, y=t, ax=ax, order=['SP', 'CHI'], palette=['#d95f02', '#1b9e77'], hue=city, legend=False)
    ax.axhline(t[TARGET], color='red', linestyle='--', linewidth=1)
    ax.set_title(f"{c}\n({lv['hybrid'][c]} in hybrid)", fontsize=9); ax.set_xlabel(''); ax.set_ylabel('')
axes[-1].set_axis_off()
fig.suptitle('Transformed feature values by city (red line: Brás)'); fig.tight_layout()
save(fig, 'city_levels_boxplots'); plt.show()

## 2. Scaling (hybrid)

Standardized values beyond ±3 before the cap, by column; for relative columns the mean and SD are the unit's own city's.

In [ ]:
rows = []
for c in scalar_cols:
    t = cm.transform(df[c], cm.TRANSFORM[c])
    g = t.groupby(city) if lv['hybrid'][c] == 'relative' else None
    z = ((t - g.transform('mean')) / g.transform(lambda s: s.std(ddof=0))) if g is not None else (t - t.mean()) / t.std(ddof=0)
    rows.append(dict(column=c, transform=cm.TRANSFORM[c], level=lv['hybrid'][c], units_capped=int((z.abs() > cm.CAP).sum()),
                     capped=', '.join(label(u) for u in z.index[z.abs() > cm.CAP])))
sc = pd.DataFrame(rows).set_index('column'); sc.to_csv(TAB / 'scaling_hybrid.csv'); display(sc)

## 3. Run R1 — equal weights (hybrid)

In [ ]:
def build(scaling, fams=None, method='standard', calibration='median', frame=None):
    fams = fams or families
    return cm.blocks(frame if frame is not None else xm.frame_for(df, subs, scaling), fams, method, calibration, xm.levels(scaling))

def rankings(D):
    r = cm.rank_target(D, ids, TARGET); r['name'] = names; r['city'] = city
    r['rank_in_city'] = r.groupby('city').distance.rank(method='min').astype(int)
    return r

bl = build('hybrid'); bud = cm.equal_budgets(families)
D1, contrib1 = cm.distance(bl, bud)
pd.DataFrame(D1, index=ids, columns=ids).to_csv(TAB / 'distance_hybrid_R1.csv')
display(pd.DataFrame([dict(family=b['family'], block=b['block'], columns=len(b['columns']), budget=bud[b['family']] * b['share'],
                           calibration=b['cal']) for b in bl]).round(4))
res = rankings(D1)
res.to_csv(TAB / 'ranking_bras_hybrid_R1.csv')
nb1 = cm.neighbours(D1, ids); nb1['label'] = nb1.unit_id.map(label); nb1['neighbour_label'] = nb1.neighbour.map(label)
nb1.to_csv(TAB / 'neighbours_hybrid_R1.csv', index=False)
print("Top 10 most similar units to Brás (all 172):"); display(res[['name', 'city', 'distance', 'rank']].head(10))
print("Top 10 most similar CHICAGO Community Areas to Brás:"); display(res[res.city == 'CHI'][['name', 'distance', 'rank', 'rank_in_city']].head(10))
print("Top 10 most similar São Paulo districts to Brás:"); display(res[res.city == 'SP'][['name', 'distance', 'rank', 'rank_in_city']].head(10))

### Which families drive the distances (all pairs)
What each family actually contributes to $D^2$ across all 14,878 pairs; the nominal equal share is 1/12.

In [ ]:
shares = cm.contribution_shares(contrib1, D1); shares.round(3).to_csv(TAB / 'family_contribution_shares_hybrid_R1.csv')
display(shares.round(3))
fig, ax = plt.subplots(figsize=(9, 4))
shares.mean_share.plot(kind='bar', color='steelblue', ax=ax); ax.axhline(1 / 12, color='r', linestyle='--', label='nominal 1/12')
ax.set_ylabel('Mean share of D² over all pairs'); ax.set_title('Hybrid R1: actual contribution of each family'); ax.legend()
save(fig, 'family_contribution_shares'); plt.show()

## Visualizing results
### Breakdown of squared distance for the top 10 neighbours
First across all units, then for the 10 Chicago areas closest to Brás.

In [ ]:
ti = ids.index(TARGET)
def contrib_plot(units, title, name):
    plot = pd.DataFrame({f: [contrib1[f][ti, ids.index(d)] for d in units] for f in families}, index=[label(d) for d in units])
    fig, ax = plt.subplots(figsize=(12, 6))
    plot.plot(kind='bar', stacked=True, colormap='tab20', ax=ax)
    ax.set_title(title); ax.set_ylabel('Squared distance (total is D²)'); ax.set_xlabel('')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left'); plt.xticks(rotation=45, ha='right')
    save(fig, name); plt.show()
top_10 = res.head(10).index; top_chi = res[res.city == 'CHI'].head(10).index
contrib_plot(top_10, 'Squared distance contributions for the top 10 neighbours of Brás (hybrid R1)', 'contributions_top10_all')
contrib_plot(top_chi, 'Squared distance contributions for the 10 Chicago areas closest to Brás (hybrid R1)', 'contributions_top10_chicago')

### Worked example: Brás and its closest Chicago match, family by family

In [ ]:
j = ids.index(top_chi[0])
we = pd.DataFrame({'contribution_to_D2': {f: contrib1[f][ti, j] for f in families}})
we['share'] = we.contribution_to_D2 / we.contribution_to_D2.sum(); we.loc['total'] = [we.contribution_to_D2.sum(), 1.0]
print(f"Brás vs {label(top_chi[0])}: D = sqrt({we.loc['total', 'contribution_to_D2']:.4f}) = {D1[ti, j]:.4f}")
we.round(4).to_csv(TAB / 'worked_example_bras_vs_top_chicago.csv'); display(we.round(4))

In [ ]:
gsp, gch = xm.geometry()
for g in (gsp, gch):
    g['name'] = g.unit_id.map(names)
gsp = gsp.merge(res[['distance', 'rank', 'rank_in_city']], left_on='unit_id', right_index=True, how='left')
gch = gch.merge(res[['distance', 'rank', 'rank_in_city']], left_on='unit_id', right_index=True, how='left')

m = gsp.explore(tiles='OpenStreetMap', tooltip=['name', 'distance', 'rank_in_city'],
                style_kwds={'weight': 1, 'color': 'gray', 'fillOpacity': 0.1, 'fillColor': '#e0e0e0'})
m = gsp[gsp.rank_in_city <= 10].explore(m=m, style_kwds={'weight': 3, 'fillOpacity': 0.4, 'color': 'red', 'fillColor': 'red'}, tooltip=['name', 'rank_in_city', 'distance'])
m = gsp[gsp.unit_id == TARGET].explore(m=m, style_kwds={'weight': 4, 'fillOpacity': 0.4, 'color': 'blue', 'fillColor': 'blue'}, tooltip=['name'])
m

In [ ]:
m = gch.explore(tiles='OpenStreetMap', tooltip=['name', 'distance', 'rank_in_city'],
                style_kwds={'weight': 1, 'color': 'gray', 'fillOpacity': 0.1, 'fillColor': '#e0e0e0'})
m = gch[gch.rank_in_city <= 10].explore(m=m, style_kwds={'weight': 3, 'fillOpacity': 0.4, 'color': 'red', 'fillColor': 'red'}, tooltip=['name', 'rank_in_city', 'distance'])
m

In [ ]:
# Static maps (saved for the report): same colour scale in both cities; each city's top 10 outlined.
vmin, vmax = np.nanmin(res.distance), np.nanmax(res.distance)
fig, axes = plt.subplots(1, 2, figsize=(15, 9))
for ax, g, crs, title in ((axes[0], gsp, 31983, 'São Paulo'), (axes[1], gch, 26916, 'Chicago')):
    g = g.to_crs(crs)
    g.plot(column='distance', cmap='viridis_r', vmin=vmin, vmax=vmax, ax=ax, missing_kwds={'color': 'navy'})
    g[g.rank_in_city <= 10].boundary.plot(ax=ax, color='red', linewidth=1.6)
    if (g.unit_id == TARGET).any():
        g[g.unit_id == TARGET].plot(ax=ax, color='navy')
    ax.set_axis_off(); ax.set_title(title)
sm = plt.cm.ScalarMappable(cmap='viridis_r', norm=plt.Normalize(vmin, vmax))
fig.colorbar(sm, ax=axes, shrink=.6, label='Hybrid R1 distance to Brás')
fig.suptitle('Distance to Brás (navy); each city\'s 10 closest units outlined in red')
save(fig, 'maps_distance_to_bras'); plt.show()

### Radar chart comparison: Brás vs its closest Chicago match
The model's own standardized values (hybrid: absolute columns on the 173-unit scale, relative columns on each city's scale), one per family; U6 by its intensity.

In [ ]:
Z = xm.scaled(df, families, 'hybrid')
top_match_id = top_chi[0]
radar_cols = {'M1: Street dens': 'm1_km_per_km2', 'M3: Block size': 'ov_m3_wmedian_ln_m2', 'M4: Compactness': 'ov_m4_wmedian_compactness',
              'M6: Major share': 'm6_major_share', 'M7: Parcel dens': 'm7_parcels_per_km2', 'B1: Bldg cover': 'B1_coverage_land',
              'U3: Pop dens': 'u3_acs_land_km2', 'BV: Height (rel.)': 'bv_height_built_m', 'U2: Job dens (rel.)': 'u2_jobs_land_km2',
              'U4: Transit (rel.)': 'u4_ptai_avg_resident', 'U6: Activity (rel.)': 'u6_log_intensity', 'U1: Res. land (rel.)': 'p_residential'}
cats = list(radar_cols); N = len(cats)
v1 = Z.loc[TARGET, list(radar_cols.values())].tolist(); v2 = Z.loc[top_match_id, list(radar_cols.values())].tolist()
v1 += v1[:1]; v2 += v2[:1]
ang = [k / float(N) * 2 * pi for k in range(N)]; ang += ang[:1]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2); ax.set_theta_direction(-1); plt.xticks(ang[:-1], cats, size=10); ax.set_rlabel_position(0)
ax.set_ylim(-3.5, 3.5)
ax.plot(ang, v1, linewidth=2, label='Brás (reference)'); ax.fill(ang, v1, alpha=.1)
ax.plot(ang, v2, linewidth=2, label=f'{label(top_match_id)} (top Chicago match)'); ax.fill(ang, v2, alpha=.1)
ax.set_title(f'Standardized profile (hybrid): Brás vs {label(top_match_id)}', size=14, y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
save(fig, 'radar_bras_vs_top_chicago'); plt.show()

## 4. Principal component analysis (PCA) and attribute variance

The model is unsupervised, so feature importance comes from the structure of the data: PCA of the hybrid R1 geometry over 173 units shows which directions carry the variation across both cities. Loadings are correlations of each component with the model columns.

In [ ]:
coords = cm.coordinates(bl, ids)
P = cm.pca(D1, ids, coords); pca_scores = P['scores']; loadings = P['loadings']
print(f"Negative-eigenvalue mass (from the three units without U1): {P['negative_mass']:.2e}")
k_all = len(P['cumulative'])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, k_all + 1), P['cumulative'], marker='o', linestyle='--'); ax.axhline(0.90, color='r'); ax.text(1, .92, '90% variance threshold', color='red')
ax.set_title('PCA cumulative explained variance (hybrid R1, 173 units)'); ax.set_xlabel('Number of components'); ax.set_ylabel('Cumulative explained variance')
ax.set_xticks(range(1, k_all + 1)); save(fig, 'pca_cumulative_variance'); plt.show()
pd.DataFrame({'explained': P['explained'], 'cumulative': P['cumulative']}, index=pca_scores.columns).to_csv(TAB / 'pca_variance.csv')
loadings.to_csv(TAB / 'pca_loadings.csv')
print("PCA loadings (correlation with the component) — PC1:")
display(loadings.sort_values(by='PC1', key=abs, ascending=False)['PC1'].to_frame().head(10).round(3))

### PCA 2D visualization
All 173 units on the first two components, coloured by city; Brás and its 10 closest Chicago areas highlighted.

In [ ]:
pca_df = pca_scores[['PC1', 'PC2']].copy(); pca_df['city'] = city
fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='city', palette={'SP': '#fdae6b', 'CHI': '#a1d99b'}, s=50, ax=ax)
sns.scatterplot(data=pca_df.loc[top_chi], x='PC1', y='PC2', color='green', s=110, edgecolor='black', label='10 closest Chicago areas', ax=ax)
sns.scatterplot(data=pca_df.loc[[TARGET]], x='PC1', y='PC2', color='red', s=280, marker='*', edgecolor='black', label='Brás', ax=ax)
for u in [TARGET, top_chi[0], 'CHI:32', 'SP:78']:
    ax.text(pca_df.loc[u, 'PC1'] + .05, pca_df.loc[u, 'PC2'] + .05, label(u), weight='bold', fontsize=9)
ax.set_title('Units projected onto PC1 and PC2 (hybrid R1)'); save(fig, 'pca_scatter'); plt.show()

In [ ]:
def color_shift(val):
    if pd.isna(val):
        return ''
    return f"color: {'green' if val > 0 else 'red' if val < 0 else 'black'}"

baseline_ranks = res['rank']
def truncated(k):
    s_ = pca_scores.iloc[:, :k].to_numpy()
    d = np.linalg.norm(s_ - s_[ti], axis=1)
    r = pd.DataFrame({'pca_distance': d, 'name': names.values, 'city': city.values}, index=ids).drop(TARGET).sort_values('pca_distance', kind='stable')
    r['pca_rank'] = r.pca_distance.rank(method='min').astype(int); r['rank_shift'] = baseline_ranks - r.pca_rank
    t = r[['name', 'city', 'pca_distance', 'pca_rank', 'rank_shift']].head(10).copy(); t['pca_distance'] = t.pca_distance.round(4)
    return t

### Truncated PCA distance: k=1 dimension

In [ ]:
display(truncated(1).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=2 dimensions

In [ ]:
display(truncated(2).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=3 dimensions

In [ ]:
display(truncated(3).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=4 dimensions

In [ ]:
display(truncated(4).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=5 dimensions

In [ ]:
display(truncated(5).style.map(color_shift, subset=['rank_shift']))

### Truncated PCA distance: k=6 dimensions

In [ ]:
display(truncated(6).style.map(color_shift, subset=['rank_shift']))

## 5. Run R2 — PCA weights

Euclidean distance on the components that explain 90% of the variance. PCA is fitted to the already equal-family-weighted geometry, so it re-weights directions, not the original assumptions.

### Interpreting the PCA axes

In [ ]:
for pc, title in (('PC1', 'Dominant structural axis'), ('PC2', 'Secondary structural axis')):
    l = loadings[pc].sort_values(ascending=False)
    print(f"--- INTERPRETING {pc} ({title}) ---"); print("Strongest positive drivers:"); print(l.head(3).round(3).to_string())
    print("Strongest negative drivers:"); print(l.tail(3).sort_values().round(3).to_string()); print()
k = P['k']; D2 = P['D']
print(f"Retaining {k} principal components to capture >= 90% of the variance ({P['cumulative'][k - 1]:.1%}).")
pd.DataFrame(D2, index=ids, columns=ids).to_csv(TAB / 'distance_hybrid_R2.csv')
res2 = rankings(D2); res2.to_csv(TAB / 'ranking_bras_hybrid_R2.csv')
print("\n--- TOP 10 CHICAGO MATCHES TO BRÁS USING PCA-WEIGHTED DISTANCE (R2) ---")
display(res2[res2.city == 'CHI'][['name', 'distance', 'rank', 'rank_in_city']].head(10))
cmp2 = res2[res2.city == 'CHI'][['name', 'rank_in_city']].head(10).rename(columns={'rank_in_city': 'pca_chicago_rank'})
cmp2['equal_weight_chicago_rank'] = res.loc[cmp2.index, 'rank_in_city']; cmp2['rank_shift'] = cmp2.equal_weight_chicago_rank - cmp2.pca_chicago_rank
print("Comparison of Chicago ranks (equal weights vs PCA):"); display(cmp2.style.map(color_shift, subset=['rank_shift']))
print("Top 10 across all units (R2):"); display(res2[['name', 'city', 'distance', 'rank']].head(10))

## 6. Run R3 — without correlated features (hybrid)

Spearman among the hybrid-scaled scalar columns over the 173 units; columns are dropped while any pair has |ρ| ≥ 0.70. The U6 composition block stays.

In [ ]:
def r3(scaling):
    Zs = xm.scaled(xm.frame_for(df, subs, scaling), families, scaling)
    fam3, log = cm.prune(Zs, families)
    return fam3, log, cm.distance(build(scaling, fam3), cm.equal_budgets(fam3))[0]

fam3, prune_log, D3 = r3('hybrid')
display(prune_log); prune_log.to_csv(TAB / 'r3_pruning_log_hybrid.csv', index=False)
print("R3 families:", list(fam3))
pd.DataFrame(D3, index=ids, columns=ids).to_csv(TAB / 'distance_hybrid_R3.csv')
res3 = rankings(D3); res3.to_csv(TAB / 'ranking_bras_hybrid_R3.csv')
print("\n--- TOP 10 CHICAGO MATCHES TO BRÁS WITHOUT CORRELATED FEATURES (R3) ---")
display(res3[res3.city == 'CHI'][['name', 'distance', 'rank', 'rank_in_city']].head(10))
cmp3 = res3[res3.city == 'CHI'][['name', 'rank_in_city']].head(10).rename(columns={'rank_in_city': 'r3_chicago_rank'})
cmp3['equal_weight_chicago_rank'] = res.loc[cmp3.index, 'rank_in_city']; cmp3['rank_shift'] = cmp3.equal_weight_chicago_rank - cmp3.r3_chicago_rank
display(cmp3.style.map(color_shift, subset=['rank_shift']))

## 7. Scale comparison: all-absolute and all-relative

The two extra steps. The same three runs are repeated with every column compared on absolute levels, then with every column compared within its own city. The tables show Brás's closest Chicago areas under each scaling and run, and how far each result agrees with the hybrid R1. The last diagnostic counts, for every unit, how many of its five nearest neighbours come from the other city: absolute levels tend to keep the cities apart, relative levels tend to mix them.

In [ ]:
runs = {('hybrid', 'R1'): D1, ('hybrid', 'R2'): D2, ('hybrid', 'R3'): D3}
logs = {'hybrid': prune_log}
for s_ in ('all_absolute', 'all_relative'):
    b_ = build(s_); d1 = cm.distance(b_, bud)[0]
    runs[(s_, 'R1')] = d1
    runs[(s_, 'R2')] = cm.pca(d1, ids)['D']
    f3, lg, d3 = r3(s_); runs[(s_, 'R3')] = d3; logs[s_] = lg
for (s_, r_), Dx in runs.items():
    pd.DataFrame(Dx, index=ids, columns=ids).to_csv(TAB / f'distance_{s_}_{r_}.csv')
    rankings(Dx).to_csv(TAB / f'ranking_bras_{s_}_{r_}.csv')
for s_ in ('all_absolute', 'all_relative'):
    print(f"R3 pruning under {s_}:"); display(logs[s_]); logs[s_].to_csv(TAB / f'r3_pruning_log_{s_}.csv', index=False)

top5 = pd.DataFrame({f'{s_} {r_}': [names[u] for u in rankings(Dx).query("city == 'CHI'").index[:5]] for (s_, r_), Dx in runs.items()}, index=range(1, 6))
top5.to_csv(TAB / 'bras_top5_chicago_all_runs.csv'); print("Brás's five closest Chicago areas, every scaling and run:"); display(top5.T)

base_chi = res.loc[chi, 'distance']
rows = []
for (s_, r_), Dx in runs.items():
    rr = rankings(Dx)
    rows.append(dict(scaling=s_, run=r_, **cm.compare(Dx, D1),
                     bras_chicago_spearman=float(rr.loc[chi, 'distance'].corr(base_chi, method='spearman')),
                     bras_chicago_top10_kept=len(set(rr.query("city == 'CHI'").index[:10]) & set(top_chi)),
                     mean_other_city_neighbours=float(xm.other_city_share(Dx, ids).mean()),
                     bras_other_city_in_top5=float(xm.other_city_share(Dx, ids)[TARGET])))
scale_cmp = pd.DataFrame(rows).round(3); scale_cmp.to_csv(TAB / 'scale_comparison.csv', index=False)
display(scale_cmp)

fig, ax = plt.subplots(figsize=(9, 4))
sc_ = scale_cmp[scale_cmp.run == 'R1'].set_index('scaling')
sc_.mean_other_city_neighbours.plot(kind='bar', color=['#4c72b0', '#dd8452', '#55a868'], ax=ax)
ax.set_ylabel('Mean share of the 5 nearest from the other city'); ax.set_title('How much each scaling mixes the two cities (R1)')
plt.xticks(rotation=0); save(fig, 'scale_city_mixing'); plt.show()

## 8. Stability and sensitivities (hybrid R1)

**Weight draws.** 500 draws multiply every family budget by a uniform factor in [0.5, 1.5] (seed 20261006). A neighbour is **stable** if it stays in the top five in at least 80% of draws. The same draws test Brás's five closest Chicago areas.

**Sensitivities.** Each scenario against the hybrid R1: agreement over all pairs, shared top five over all units, and how many of Brás's five closest Chicago areas survive. All are pre-registered except the one marked POST-HOC (calibration by the mean pairwise value), kept from the Chicago model as a diagnostic.

In [ ]:
stab = cm.weight_draws(bl, families, bud, ids)
stab['label'] = stab.unit_id.map(label); stab['neighbour_label'] = stab.neighbour.map(label)
stab.to_csv(TAB / 'stability_hybrid_R1_weight_draws.csv', index=False)
print(f"Stable top-five relations: {stab.stable.mean():.1%} of {len(stab)}; units with all five stable: {stab.groupby('unit_id').stable.all().sum()} of {n}.")
bras_stab = xm.reference_stability(bl, families, bud, ids, TARGET, chi)
bras_stab['name'] = bras_stab.match.map(names); bras_stab.to_csv(TAB / 'stability_bras_top5_chicago.csv', index=False)
print("Brás's five closest Chicago areas under 500 weight draws:"); display(bras_stab[['rank', 'name', 'share_of_draws', 'stable']])
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(stab.share_of_draws, bins=20, color='steelblue'); ax.axvline(cm.STABLE, color='r', linestyle='--', label='80% rule')
ax.set_xlabel('Share of draws a baseline top-5 neighbour stays in the top 5'); ax.set_ylabel('Neighbour relations'); ax.legend()
ax.set_title('Stability of hybrid R1 neighbours under 500 weight draws'); save(fig, 'stability_histogram'); plt.show()

In [ ]:
def D_of(fams=None, method='standard', calibration='median', frame=None, budgets=None):
    f_ = fams or families
    return cm.distance(build('hybrid', f_, method, calibration, frame), budgets or cm.equal_budgets(f_))[0]

scen = {f'without {f}': D_of({k: v for k, v in families.items() if k != f}) for f in families}
scen['equal domain budgets'] = cm.distance(bl, cm.domain_budgets(families))[0]
scen['robust median/IQR scaling'] = D_of(method='robust')
scen['rank-normal scaling'] = D_of(method='rank')
scen['Ledoit-Wolf Mahalanobis (no U1)'] = cm.mahalanobis(bl)
for name, (fam, old, new) in cm.SUBSTITUTES.items():
    f2 = {k: [new if c == old else c for c in v] for k, v in families.items()}
    scen[name] = D_of(f2, frame=df.join(subs[[new]]))
u6c = xm.u6_confidence_filtered(ids)
scen['U6 confidence >= 0.3'] = D_of(frame=df.drop(columns=u6c.columns).join(u6c))
scen['POST-HOC: mean calibration'] = D_of(calibration='mean')
top5_chi = set(top_chi[:5])
sens = pd.DataFrame([dict(scenario=k, **cm.compare(v, D1), bras_top5_chicago_kept=len(top5_chi & set(rankings(v).query("city == 'CHI'").index[:5])))
                     for k, v in scen.items()]).round(3)
sens.to_csv(TAB / 'sensitivities_hybrid.csv', index=False); display(sens)

## 9. Descriptive check: Ward hierarchical clustering

Ward clustering on the hybrid R1 geometry (principal coordinates) over 173 units. A descriptive consistency check, not an independent validation of urban types. The cluster-by-city table shows whether the clusters cross the two cities.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch

E = pca_scores.to_numpy()
print("--- WARD CLUSTERING: SILHOUETTE SCORES ---")
sil = {}
for kk in range(3, 9):
    lab = AgglomerativeClustering(n_clusters=kk, linkage='ward').fit_predict(E)
    sil[kk] = silhouette_score(E, lab); print(f"k={kk} clusters: silhouette = {sil[kk]:.4f}")
best_k = max(sil, key=sil.get); print(f"\nBest tested k by silhouette (descriptive): {best_k}")
clusters = pd.Series(AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(E), index=ids, name='cluster')
pd.DataFrame({'name': names, 'city': city, 'cluster': clusters, 'rank_to_bras': res['rank']}).to_csv(TAB / 'ward_clusters.csv')
print("\nUnits per cluster and city:"); display(pd.crosstab(clusters, city))
members = pd.DataFrame({'name': names, 'city': city})[clusters == clusters[TARGET]]
members['equal_weight_rank'] = res['rank']; members.loc[TARGET, 'equal_weight_rank'] = 0
print(f"Brás is in cluster {clusters[TARGET]}. Members:"); display(members.sort_values('equal_weight_rank'))

In [ ]:
fig = plt.figure(figsize=(22, 8))
sch.dendrogram(sch.linkage(E, method='ward'), labels=[label(u) for u in ids], leaf_rotation=90, leaf_font_size=6)
plt.title('Ward hierarchical clustering dendrogram (hybrid R1, 173 units)'); plt.ylabel('Euclidean distance')
save(fig, 'ward_dendrogram'); plt.show()

pca_df['cluster'] = clusters
fig, ax = plt.subplots(figsize=(10, 8)); pal = sns.color_palette('tab10', n_colors=best_k)
for c_ in range(best_k):
    d_ = pca_df[pca_df.cluster == c_]
    ax.scatter(d_.PC1, d_.PC2, color=pal[c_], s=55, marker='o', label=f'Cluster {c_}', edgecolor=np.where(d_.city == 'CHI', 'black', 'none'))
ax.scatter(pca_df.loc[TARGET, 'PC1'], pca_df.loc[TARGET, 'PC2'], color='black', s=280, marker='*', label='Brás')
ax.set_title(f'Units on PC1 and PC2 coloured by Ward cluster (k={best_k}; black edge = Chicago)'); ax.legend()
save(fig, 'pca_clusters'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 9))
for ax, g, crs, title in ((axes[0], gsp, 31983, 'São Paulo'), (axes[1], gch, 26916, 'Chicago')):
    g = g.merge(clusters, left_on='unit_id', right_index=True).to_crs(crs)
    g.plot(column='cluster', categorical=True, cmap='tab10', vmin=0, vmax=9, legend=True, ax=ax, edgecolor='white', linewidth=.3)
    ax.set_axis_off(); ax.set_title(title)
fig.suptitle(f'Ward clusters (k={best_k}), same colours in both cities'); save(fig, 'maps_clusters'); plt.show()

## 10. Saved outputs

Tables in `analysis/results/SP_CHI/sp_chicago_model_v1_2026_10_06/tables/`, figures in `figures/`. Methodology and reading: `docs/harmonization/MODEL_REPORT.md`.

In [ ]:
print(sorted(p.name for p in TAB.glob('*.csv'))); print(sorted(p.name for p in FIG.glob('*.png')))